In [0]:

#Leyendo la tabla de caracteristicas generales
ventas = spark.read.format("delta").table("workspace.electiva.caso_practico_ventas")

# Convertir el DataFrame de Spark a Pandas
ventas = ventas.toPandas()

ventas

In [0]:
import pandas as pd

# 2. Identificación de Tipos
# Vemos cómo Pandas interpretó cada columna (texto, número entero, decimal, etc.)
print("\n--- Información general y tipos de datos ---")
ventas.info()

# 3. Detección de Nulos
# Contamos cuántas celdas vacías hay en cada columna y calculamos su porcentaje
nulos = ventas.isnull().sum()
porcentaje_nulos = (nulos / len(ventas)) * 100

print("\n--- Resumen de Valores Nulos ---")
print(pd.DataFrame({'Total Nulos': nulos, 'Porcentaje (%)': porcentaje_nulos}))

# 4. Estadística Descriptiva
# Analizamos el comportamiento estadístico básico de las columnas numéricas solicitadas
print("\n--- Estadísticas de Cantidad y Precio ---")
print(ventas[['cantidad', 'precio_unitario']].describe())

Fase 2 - Limpieza

In [0]:
# 1. Tratamiento de Duplicados
# Eliminamos registros donde el 'id_transaccion' sea idéntico, conservando solo el primero
ventas = ventas.drop_duplicates(subset=['id_transaccion'], keep='first')

# 2. Normalización de Texto
# Arreglamos el error de escritura en los métodos de pago
ventas['metodo_pago'] = ventas['metodo_pago'].replace({'Tarjeta Credito': 'Tarjeta de Crédito'})

# 3. Gestión de Valores Nulos
# Imputamos (rellenamos) la cantidad vacía con la mediana de la columna
mediana_cant = ventas['cantidad'].median()
ventas['cantidad'] = ventas['cantidad'].fillna(mediana_cant)

# Imputamos la satisfacción vacía con la moda (el valor más repetido)
moda_satisfaccion = ventas['satisfaccion_cliente'].mode()[0]
ventas['satisfaccion_cliente'] = ventas['satisfaccion_cliente'].fillna(moda_satisfaccion)

# 4. Tratamiento de Atípicos (Outliers)
# Localizamos la venta con el precio ilógico de 9999
outlier_condicion = ventas['precio_unitario'] == 9999

# Calculamos el precio promedio de la categoría 'Ropa' (excluyendo el dato erróneo)
promedio_ropa = ventas[(ventas['categoria_producto'] == 'Ropa') & (~outlier_condicion)]['precio_unitario'].mean()

# Reemplazamos el 9999 por ese promedio calculado
ventas.loc[outlier_condicion, 'precio_unitario'] = promedio_ropa

Fase 3 - Transformación e Ingeniería de Atributos

In [0]:
import numpy as np

# 1. Columna Derivada
# Calculamos los ingresos totales de cada transacción
ventas['monto_total'] = ventas['cantidad'] * ventas['precio_unitario']

# 2. Conversión Temporal
# Transformamos el texto genérico a un formato oficial de fecha temporal
ventas['fecha'] = pd.to_datetime(ventas['fecha'])

# Extraemos el día de la semana y el número del mes como nuevas características
ventas['dia_semana'] = ventas['fecha'].dt.day_name()
ventas['mes'] = ventas['fecha'].dt.month

# 3. Discretización (Binning)
# Definimos los cortes de los rangos de precios
limites = [0, 100, 500, np.inf]
etiquetas = ['Bajo', 'Medio', 'Alto']

# Segmentamos la variable continua en las categorías discretas
ventas['rango_monto'] = pd.cut(ventas['monto_total'], bins=limites, labels=etiquetas)

Fase 4: Agregación y Resumen de Datos

In [0]:
# 1. Agrupamiento Simple
# Total de ingresos y promedio de unidades por cada línea de producto
resumen_productos = ventas.groupby('categoria_producto').agg({
    'monto_total': 'sum',
    'cantidad': 'mean'
}).reset_index()

# 2. Tabla Dinámica
# Resumen en matriz: Regiones vs Categorías, sumando el dinero total
matriz_ventas = pd.pivot_table(
    ventas, 
    values='monto_total', 
    index='region', 
    columns='categoria_producto', 
    aggfunc='sum',
    fill_value=0
)

# 3. Análisis de Satisfacción
# Nivel de satisfacción promedio dependiendo del medio de pago
satisfaccion_pagos = ventas.groupby('metodo_pago')['satisfaccion_cliente'].mean().reset_index()

Fase 5: Selección, Filtrado y Muestreo

In [0]:
# 1. Filtrado Multicriterio
# Creamos una máscara booleana con las dos condiciones solicitadas
filtro_norte = (ventas['region'] == 'Norte') 
filtro_satisfaccion = (ventas['satisfaccion_cliente'] >= 4)

# Aplicamos los filtros a la tabla original usando el operador & (Y)
ventas_norte_exitosas = ventas[filtro_norte & filtro_satisfaccion]

# 2. Muestreo Estratificado
# Extraemos el 10% de los datos totales, pero agrupados por región para no perder la proporción
muestra = ventas.groupby('region').sample(frac=0.10)